In [ ]:
###Feature Basic Filtering

In [ ]:
import pandas as pd
import seaborn as sns
import pathlib
from ALLCools.mcds import MCDS
import dask
import anndata
import scanpy as sc
from ALLCools.clustering import cluster_enriched_features, significant_pc_test, log_scale, ConsensusClustering, Dendrogram, get_pc_centers
import numpy as np
import matplotlib.pyplot as plt
from ALLCools.clustering import tsne, balanced_pca, filter_regions, remove_black_list_region, lsi, binarize_matrix
from ALLCools.plot import *
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from sklearn.metrics import pairwise_distances

sns.set_context(context='notebook', font_scale=1.3)

In [ ]:
# change this to the path to your metadata
metadata_path = 'CellMetadata.PassQC.Quintile.csv.gz'

# change this to the paths to your MCDS files, 
mcds_path = './GSC_epitherapy_CTCF_new1.mcds/'

# Basic filtering parameters. 
mapping_rate_cutoff = 50
mapping_rate_col_name = 'R1MappingRate'  # Name may change
final_reads_cutoff = 500000
final_reads_col_name = 'FinalmCReads'  # Name may change
mccc_cutoff = 0.03
mccc_col_name = 'mCCCFrac'  # Name may change
mch_cutoff = 0.2
mch_col_name = 'mCHFrac'  # Name may change
mcg_cutoff = 0.05
mcg_col_name = 'mCGFrac'  # Name may change


# Dimension name used to do clustering
obs_dim = 'cell'  # observation
var_dim = 'CTCF_scaled'  # feature

# feature cov cutoffs
min_cov = 10
max_cov = 150

# Regions to remove during the clustering analysis
black_list_path = '/scratch/devtools/mmoore/genomes/snm3c/hg38/hg38-blacklist.v2.bed.gz'
black_list_fraction = 0.2
f = 0.2
exclude_chromosome = ['chrM', 'chrY']

# load to memory or not
load = True

# HVF
mch_pattern = 'CHN'
mcg_pattern = 'CGN'
n_top_feature = 20000

# PC cutoff
pc_cutoff = 0.1

# KNN
knn = -1  # -1 means auto determine

# Leiden
resolution = 1

In [ ]:
metadata = pd.read_csv(metadata_path, index_col=0)
print(f'Metadata of {metadata.shape[0]} cells')
metadata.head()

In [ ]:
mcds = MCDS.open(mcds_path, 
                 var_dim=var_dim, 
                 use_obs=metadata.index)
total_feature = mcds.get_index(var_dim).size

In [ ]:
mcds

In [ ]:
# you can add the cell metadata into MCDS
mcds.add_cell_metadata(metadata)

In [ ]:
mcds.add_feature_cov_mean()

In [ ]:
mcds = mcds.filter_feature_by_cov_mean(
    min_cov=min_cov,  # minimum coverage
    max_cov=max_cov  # Maximum coverage
)
mcds

In [ ]:
mcds = mcds.remove_black_list_region(
    black_list_path=black_list_path,
    f=f  # Features having overlap > f with any black list region will be removed.
)

In [ ]:
mcds = mcds.remove_chromosome(exclude_chromosome)

In [ ]:
print(
    f'{mcds.get_index(var_dim).size} ({mcds.get_index(var_dim).size * 100 / total_feature:.1f}%) '
    f'{var_dim} remained after all the basic filter.')

In [ ]:
mcds.add_mc_frac(
normalize_per_cell=False  # after calculating mC frac, per cell normalize the matrix
    #clip_norm_value=10  # clip outlier values above 10 to 10
)

# load only the mC fraction matrix into memory so following steps is faster
# Only load into memory when your memory size is enough to handle your dataset
if load and (mcds.get_index(obs_dim).size < 20000):
    mcds[f'{var_dim}_da_frac'].load()

In [ ]:
mcds

In [ ]:
da = mcds['CTCF_scaled_da_frac'].values

In [ ]:
data = da[:, :, 0]
data

In [ ]:
cells = mcds['cell'].values.astype(str)
cells

In [ ]:
df = pd.DataFrame(data, index=cells, columns=mcds['CTCF_scaled'].values)
df

In [ ]:
df_avg = pd.DataFrame({
    "non": df.filter(regex="^non_").mean(axis=1),
    "down": df.filter(regex="^down_").mean(axis=1),
    "up": df.filter(regex="^up_").mean(axis=1),
}, index=df.index)

#df_avg["group"] = (
#    df_avg.index
#      .str.split("_")
#      .str[:2]
#      .str.join("_")
#)

df_avg["group"] = mcds['cell_qcluster'].values

df_avg

In [ ]:
df_long = (
    df_avg
    .reset_index()
    .melt(
        id_vars=["index", "group"],
        var_name="type",
        value_name="value"
    )
)
df_long

In [ ]:
leg=['B36_0','B36_1','B36_2','B36_3','B36_4','B36_5','B36_6','B49_0','B49_1','B49_2','B49_3','B49_4','B49_5','B49_6',
     'B66_0','B66_1','B66_2','B66_3','B66_4','B66_5','B66_6','NHA_0','NHA_1','NHA_2','NHA_3','NHA_4','NHA_5','NHA_6','qNHA_0','qNHA_1','qNHA_2','qNHA_3','qNHA_4','qNHA_5','qNHA_6']
ax = sns.boxplot(data=df_long, x='group', y='value', hue='type', order=leg)
ax.set_xticklabels(ax.get_xticklabels(), rotation=90, horizontalalignment='left')
sns.move_legend(ax, "center left", bbox_to_anchor=(1, 0.5), title="mCG Group")
ax

In [ ]:
df_long["cell"] = df_long["index"].str.split("_").str[0]
df_long["treatment"] = df_long["index"].str.split("_").str[1]
df_long

In [ ]:
df_long.to_csv("cell_peak_avg_mCG.tsv", sep="\t", index=False)

In [ ]:
with open('FeatureList.BasicFilter.txt', 'w') as f:
    for var in mcds.get_index(var_dim).astype(str):
        f.write(var + '\n')

In [ ]:
mcds

In [ ]:
###Calculate Highly Variable Features And Get mC Fraction AnnData

In [ ]:
import pathlib
import pandas as pd
import dask
from ALLCools.mcds import MCDS

In [ ]:
# If True, will load all data into memory.
# Computation will be much faster, but also very memory intensive, only use this for small number of cells (<10,000)
load = True

# change this to the path to your metadata
metadata_path = 'CellMetadata.PassQC.csv.gz'

# change this to the paths to your MCDS files, 
mcds_path = './GSC_epitherapy_with5k.mcds/'

# Feature list after basic filter
feature_path = 'FeatureList.BasicFilter.txt'

# Dimension name used to do clustering
obs_dim = 'cell'  # observation
var_dim = 'chrom5k'  # feature

# HVF method:
# SVR: regression based
# Bins: normalize dispersion per bin
hvf_method = 'SVR'
mch_pattern = 'CHN'
mcg_pattern = 'CGN'
n_top_feature = 20000

# Downsample cells
downsample = 20000

In [ ]:
metadata = pd.read_csv(metadata_path, index_col=0)
total_cells = metadata.shape[0]
print(f'Metadata of {total_cells} cells')

In [ ]:
metadata.head()

In [ ]:
use_features = pd.read_csv(feature_path, header=None, index_col=0).index
use_features.name = var_dim

In [ ]:
total_mcds = MCDS.open(mcds_path,
                       var_dim=var_dim,
                       use_obs=metadata.index).sel({var_dim: use_features})

In [ ]:
total_mcds.add_mc_rate(var_dim=var_dim,
                       normalize_per_cell=False,
                       clip_norm_value=10)

total_mcds

In [ ]:
if downsample and total_cells > downsample:
    # make a downsampled mcds
    print(f'Downsample cells to {downsample} to calculate HVF.')
    downsample_cell_ids = metadata.sample(downsample, random_state=0).index
    mcds = total_mcds.sel(
        {obs_dim: total_mcds.get_index(obs_dim).isin(downsample_cell_ids)})
else:
    mcds = total_mcds

In [ ]:
if load and (mcds.get_index('cell').size <= 20000):
    # load the relevant data so the computation can be fater, watch out memory!
    mcds[f'{var_dim}_da_frac'].load()

In [ ]:
if hvf_method == 'SVR':
    # use SVR based method
    mch_hvf = mcds.calculate_hvf_svr(var_dim=var_dim,
                                     mc_type=mch_pattern,
                                     n_top_feature=n_top_feature,
                                     plot=True)
else:
    # use bin based method
    mch_hvf = mcds.calculate_hvf(var_dim=var_dim,
                                 mc_type=mch_pattern,
                                 min_mean=0,
                                 max_mean=5,
                                 n_top_feature=n_top_feature,
                                 bin_min_features=5,
                                 mean_binsize=0.05,
                                 cov_binsize=100)

In [ ]:
total_mcds.coords[f'{var_dim}_{mch_pattern}_feature_select'] = mcds.coords[
    f'{var_dim}_{mch_pattern}_feature_select']

In [ ]:
mch_adata = total_mcds.get_adata(mc_type=mch_pattern,
                           var_dim=var_dim,
                           select_hvf=True)

mch_adata.write_h5ad(f'mCH.HVF.h5ad')

mch_adata

In [ ]:
if hvf_method == 'SVR':
    # use SVR based method
    mcg_hvf = mcds.calculate_hvf_svr(var_dim=var_dim,
                                     mc_type=mcg_pattern,
                                     n_top_feature=n_top_feature,
                                     plot=True)
else:
    # use bin based method
    mcg_hvf = mcds.calculate_hvf(var_dim=var_dim,
                                 mc_type=mcg_pattern,
                                 min_mean=0,
                                 max_mean=5,
                                 n_top_feature=n_top_feature,
                                 bin_min_features=5,
                                 mean_binsize=0.02,
                                 cov_binsize=20)

In [ ]:
total_mcds.coords[f'{var_dim}_{mch_pattern}_feature_select'] = mcds.coords[
    f'{var_dim}_{mch_pattern}_feature_select']

In [ ]:
mcg_adata = total_mcds.get_adata(mc_type=mcg_pattern,
                                 var_dim=var_dim,
                                 select_hvf=True)

mcg_adata.write_h5ad(f'mCG.HVF.h5ad')

mcg_adata

In [ ]:
###Preclustering and Cluster Enriched Features

In [ ]:
import seaborn as sns
import anndata
import scanpy as sc
from ALLCools.clustering import cluster_enriched_features, significant_pc_test, log_scale

In [ ]:
sns.set_context(context='notebook', font_scale=1.3)

In [ ]:
adata_path = 'mCH.HVF.h5ad'

# Cluster Enriched Features analysis
top_n=200
alpha=0.05
stat_plot=True

# you may provide a pre calculated cluster version. 
# If None, will perform basic clustering using parameters below.
cluster_col = None  

# These parameters only used when cluster_col is None
k=25
resolution=1
cluster_plot=True

In [ ]:
adata = anndata.read_h5ad(adata_path)

In [ ]:
if cluster_col is None:
    # IMPORTANT
    # put the unscaled matrix in adata.raw
    adata.raw = adata
    log_scale(adata)
    
    sc.tl.pca(adata, n_comps=100)
    significant_pc_test(adata, p_cutoff=0.1, update=True)
    
    sc.pp.neighbors(adata, n_neighbors=k)
    sc.tl.leiden(adata, resolution=resolution)
    
    if cluster_plot:
        sc.tl.umap(adata)
        sc.pl.umap(adata, color='leiden')
    
    # return to unscaled X, CEF need to use the unscaled matrix
    adata = adata.raw.to_adata()
    
    cluster_col = 'leiden'

In [ ]:
cluster_enriched_features(adata,
                          cluster_col=cluster_col,
                          top_n=top_n,
                          alpha=alpha,
                          stat_plot=True)

In [ ]:
# save adata
adata.write_h5ad(adata_path)
adata

In [ ]:
###Preclustering and Cluster Enriched Features

In [ ]:
import seaborn as sns
import anndata
import scanpy as sc
from ALLCools.clustering import cluster_enriched_features, significant_pc_test, log_scale

In [ ]:
sns.set_context(context='notebook', font_scale=1.3)

In [ ]:
adata_path = 'mCG.HVF.h5ad'

# Cluster Enriched Features analysis
top_n=200
alpha=0.05
stat_plot=True

# you may provide a pre calculated cluster version. 
# If None, will perform basic clustering using parameters below.
cluster_col = None  

# These parameters only used when cluster_col is None
k=25
resolution=1
cluster_plot=True

In [ ]:
adata = anndata.read_h5ad(adata_path)

In [ ]:
if cluster_col is None:
    # IMPORTANT
    # put the unscaled matrix in adata.raw
    adata.raw = adata
    log_scale(adata)
    
    sc.tl.pca(adata, n_comps=100)
    significant_pc_test(adata, p_cutoff=0.1, update=True)
    
    sc.pp.neighbors(adata, n_neighbors=k)
    sc.tl.leiden(adata, resolution=resolution)
    
    if cluster_plot:
        sc.tl.umap(adata)
        sc.pl.umap(adata, color='leiden')
        
    # return to unscaled X, CEF need to use the unscaled matrix
    adata = adata.raw.to_adata()
    
    cluster_col = 'leiden'

In [ ]:
cluster_enriched_features(adata,
                          cluster_col=cluster_col,
                          top_n=top_n,
                          alpha=alpha,
                          stat_plot=True)

In [ ]:
# save adata
adata.write_h5ad(adata_path)
adata

In [ ]:
###Decomposition And Embedding

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scanpy as sc
import anndata

from ALLCools.clustering import tsne, balanced_pca, significant_pc_test, log_scale
from ALLCools.plot import *

In [ ]:
sns.set_context(context='notebook', font_scale=1.3)

In [ ]:
# cell metadata path
metadata_path = 'CellMetadata.PassQC.csv.gz'

# HVF mC Fraction AnnData Files
ch_adata_path = 'mCH.HVF.h5ad'
cg_adata_path = 'mCG.HVF.h5ad'

# use feature type
# HVF: all highly variable features
# CEF: cluster enriched features
feature_type = 'CEF' 
pre_cluster_name = 'leiden'

# n_components
n_components = 'auto'  # if auto, will use Kolmogorov-Smirnov test to test the adjacent PCs and cut when P > p_cutoff
p_cutoff = 0.1  # ks test p value cutoff, only apply when n_components == 'auto'

# downsample large clusters
max_cell_prop = 0.05

In [ ]:
metadata = pd.read_csv(metadata_path, index_col=0)

In [ ]:
ch_adata = anndata.read_h5ad(ch_adata_path)
cg_adata = anndata.read_h5ad(cg_adata_path)

In [ ]:
if feature_type == 'CEF':
    print('Using Cluster Enriched Features')
    ch_adata = ch_adata[:, ch_adata.var[f'{pre_cluster_name}_enriched_features']].copy()
    cg_adata = cg_adata[:, cg_adata.var[f'{pre_cluster_name}_enriched_features']].copy()

In [ ]:
ch_scaler = log_scale(ch_adata)

cg_scaler = log_scale(cg_adata)

In [ ]:
balanced_pca(ch_adata, groups=pre_cluster_name)
sc.pl.pca_variance_ratio(ch_adata)
ch_n_components = significant_pc_test(ch_adata, p_cutoff=p_cutoff)

In [ ]:
hue = 'mCHFrac'
if hue in metadata.columns:
    ch_adata.obs[hue] = metadata[hue].reindex(ch_adata.obs_names)
    fig, axes = plot_decomp_scatters(ch_adata,
                                     n_components=ch_n_components,
                                     hue=hue,
                                     hue_quantile=(0.25, 0.75),
                                     nrows=5,
                                     ncols=5)

In [ ]:
balanced_pca(cg_adata, groups=pre_cluster_name)
sc.pl.pca_variance_ratio(cg_adata)
cg_n_components = significant_pc_test(cg_adata, p_cutoff=p_cutoff)

In [ ]:
hue = 'mCGFrac'
if hue in metadata.columns:
    cg_adata.obs[hue] = metadata[hue].reindex(cg_adata.obs_names)
    fig, axes = plot_decomp_scatters(cg_adata,
                                     n_components=cg_n_components,
                                     hue=hue,
                                     hue_quantile=(0.25, 0.75),
                                     nrows=5,
                                     ncols=5)

In [ ]:
ch_pcs = ch_adata.obsm['X_pca'][:, :ch_n_components]
cg_pcs = cg_adata.obsm['X_pca'][:, :cg_n_components]

# scale the PCs so CH and CG PCs has the same total var
#cg_pcs = cg_pcs / cg_pcs.std()
#ch_pcs = ch_pcs / ch_pcs.std()

# total_pcs
total_pcs = np.hstack([ch_pcs, cg_pcs])

In [ ]:
# make a copy of adata, add new pcs
# this is suboptimal, will change this when adata can combine layer and X in the future
adata = ch_adata.copy()

In [ ]:
adata.obsm['X_pca'] = total_pcs
del adata.uns['pca']
del adata.varm['PCs']

In [ ]:
tsne(adata,
     obsm='X_pca',
     metric='euclidean',
     exaggeration=-1,  # auto determined
     perplexity=30,
     n_jobs=-1)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4), dpi=250)
_ = categorical_scatter(data=adata, ax=ax, coord_base='tsne', hue=pre_cluster_name, show_legend=True)

In [ ]:
sc.pp.neighbors(adata)
try:
    sc.tl.paga(adata, groups=pre_cluster_name)
    sc.pl.paga(adata, plot=False)
    sc.tl.umap(adata, init_pos='paga')
except:
    sc.tl.umap(adata)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4), dpi=250)
_ = categorical_scatter(data=adata, ax=ax, coord_base='umap', hue=pre_cluster_name, show_legend=True)

In [ ]:
# in order to reduce the page size, I downsample the data here, you don't need to do this
interactive_scatter(data=adata,
                    hue=pre_cluster_name,
                    coord_base='umap')

In [ ]:
adata.write_h5ad(f'adata.with_coords.h5ad')
adata

In [ ]:
from ALLCools.clustering import ReproduciblePCA

ch_rpca = ReproduciblePCA(scaler=ch_scaler, mc_type='CHN', adata=ch_adata)
ch_rpca.dump('CHN.ReproduciblePCA.lib')

cg_rpca = ReproduciblePCA(scaler=cg_scaler, mc_type='CHN', adata=cg_adata)
cg_rpca.dump('CGN.ReproduciblePCA.lib')